[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/corrections/seance1_correction.ipynb)

# Séance 3.1 — Décrire une distribution

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire pourquoi une moyenne seule est presque toujours trompeuse
- choisir entre moyenne et médiane selon la forme de la distribution
- lire un `describe()` ligne par ligne
- mesurer la dispersion avec l'écart-type et l'écart interquartile
- repérer une concentration : quelle part du total tient dans le haut du classement

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

print(cmd.shape)
cmd.head(3)

### Exercice 1 — Moyenne et médiane des quantités

> **Votre mission :**
> - Calculer la moyenne et la médiane de la colonne `qte` (nombre d'articles par commande).
> - Les arrondir à 2 décimales → `moy_qte` et `med_qte`.

In [ ]:
moy_qte = round(cmd["qte"].mean(), 2)
med_qte = round(cmd["qte"].median(), 2)

# Meme phenomene que sur les euros : la moyenne est loin au-dessus
print(moy_qte, "articles en moyenne, mediane a", med_qte)

In [ ]:
verifier("1a - moyenne des quantites", moy_qte == 330.77, "mean()")
verifier("1b - mediane des quantites", med_qte == 195.0, "median()")

### Exercice 2 — Le décompte du haut

> **Votre mission :**
> - Quelle **part** des commandes dépasse la quantité moyenne ? En % arrondi à 1 décimale → `part_sup`.
> - Comparez au 50 % que donnerait une distribution symétrique.

In [ ]:
# (serie > valeur) donne des True/False ; leur moyenne est la proportion
part_sup = round(100 * (cmd["qte"] > cmd["qte"].mean()).mean(), 1)

print(part_sup, "% des commandes depassent la quantite moyenne")

In [ ]:
verifier("2 - part au-dessus de la moyenne", part_sup == 28.7,
         "comparez chaque qte a cmd['qte'].mean(), puis faites la moyenne des True/False")

### Exercice 3 — Lire un describe()

> **Votre mission :**
> - Afficher le `describe()` de la colonne `ca`.
> - En tirer le nombre de commandes → `nb_cmd` et le montant de la plus grosse → `ca_max`.

In [ ]:
print(cmd["ca"].describe().round(2))

# Les deux valeurs se lisent aussi dans describe() : count et max
nb_cmd = len(cmd)
ca_max = cmd["ca"].max()

print(nb_cmd, "commandes | la plus grosse :", ca_max)

In [ ]:
verifier("3a - nombre de commandes", nb_cmd == 1955, "len() sur la table")
verifier("3b - plus grosse commande", ca_max == 16774.72, "max() sur la colonne ca")

### Exercice 4 — Un seuil de livraison gratuite

> **Votre mission :**
> - La direction veut offrir la livraison aux **10 % de commandes les plus grosses**.
> - À quel montant faut-il placer le seuil ? → `seuil` (arrondi à 2 décimales)

In [ ]:
# Les 10 % du HAUT commencent au quantile 0.9 : 90 % des commandes
# sont en dessous de ce montant
seuil = round(cmd["ca"].quantile(0.9), 2)

print("livraison offerte au-dela de", seuil, "euros")

In [ ]:
verifier("4 - seuil des 10 % du haut", seuil == 1146.28,
         "les 10 % du haut commencent au quantile 0.9, pas 0.1")

### Exercice 5 — La dispersion des références

> **Votre mission :**
> - Calculer l'écart interquartile de `nart` (nombre de références par commande) → `iqr_nart`.
> - Rappel : écart interquartile = quantile 0.75 − quantile 0.25.

In [ ]:
q1 = cmd["nart"].quantile(0.25)
q3 = cmd["nart"].quantile(0.75)
iqr_nart = q3 - q1

# La moitie centrale des commandes tient entre 9 et 30,5 references
print("moitie centrale des commandes :", q1, "a", q3, "references")
print("ecart interquartile :", iqr_nart)

In [ ]:
verifier("5 - ecart interquartile de nart", iqr_nart == 21.5,
         "quantile(0.75) moins quantile(0.25)")

### Exercice 6 — La concentration, version 5 %

> **Votre mission :**
> - Quelle part du chiffre d'affaires les **5 % de commandes les plus grosses** représentent-elles ?
> - En % arrondi à 1 décimale → `part_top5`.

In [ ]:
# ascending=False : les plus grosses en premier
top = cmd["ca"].sort_values(ascending=False)
n5 = int(0.05 * len(cmd))

part_top5 = round(100 * top.head(n5).sum() / top.sum(), 1)
print(n5, "commandes font", part_top5, "% du chiffre d'affaires")

In [ ]:
verifier("6 - part des 5 % du haut", part_top5 == 29.6,
         "triez du plus grand au plus petit, puis divisez par le total")

### Exercice 7 — Le tableau par pays

> **Votre mission :**
> - Construire `parpays` : par pays, l'effectif (`count`), la moyenne et la médiane du `ca`.
> - En extraire la médiane française arrondie à 2 décimales → `med_fr`.

In [ ]:
parpays = cmd.groupby("pays")["ca"].agg(["count", "mean", "median"])

# .loc[ligne, colonne] pour aller chercher une case precise
med_fr = round(parpays.loc["France", "median"], 2)
print(med_fr)

In [ ]:
verifier("7 - mediane francaise", med_fr == 361.35,
         "agg(['count', 'mean', 'median']) puis .loc['France', 'median']")

### Exercice 8 — Le pays le plus déformé

> **Votre mission :**
> - En repartant de `parpays`, garder les pays d'au moins 20 commandes.
> - Ajouter une colonne `ecart` = moyenne − médiane, puis trouver le pays où elle est la plus grande → `pays_ecart`.
> - Cet écart mesure à quel point quelques grosses commandes déforment la moyenne.

In [ ]:
# .copy() avant d'ajouter une colonne a un extrait filtre
gros = parpays.query("count >= 20").copy()
gros["ecart"] = gros["mean"] - gros["median"]

# idxmax() renvoie l'etiquette de la ligne, donc le nom du pays
pays_ecart = gros["ecart"].idxmax()
print(pays_ecart)

In [ ]:
verifier("8 - pays le plus deforme", pays_ecart == "Suede",
         "ecart = mean - median, puis idxmax() pour avoir le nom et non la valeur")

### Exercice 9 — Les jours qui n'existent pas

> **Votre mission :**
> - Compter le nombre de jours de la semaine présents dans le fichier → `nb_jours`.
> - Trouver le jour où il y a le plus de commandes → `jour_top`.
> - Un jour manque. Lequel, et pourquoi est-ce important ?

In [ ]:
nb_jours = cmd["jour"].nunique()
jour_top = cmd["jour"].value_counts().idxmax()

print(nb_jours, "jours presents | le plus charge :", jour_top)

# Le samedi est absent : l'enseigne ne traite aucune commande ce jour-la.
# Un CA "moyen par jour" divise par 7 serait donc faux de 17 %.

In [ ]:
verifier("9a - nombre de jours presents", nb_jours == 6,
         "nunique() compte les valeurs distinctes")
verifier("9b - jour le plus charge", jour_top == "jeudi",
         "value_counts() puis idxmax()")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - Le directeur commercial veut **une seule phrase** sur le panier des clients pour son comité.
> - Calculer l'écart moyenne − médiane du `ca` → `ecart_ca` (arrondi à 2 décimales).
> - Calculer le rapport écart-type / moyenne → `cv` (arrondi à 2 décimales).
> - Puis écrivez la phrase que vous lui donneriez, en commentaire.

In [ ]:
ecart_ca = round(cmd["ca"].mean() - cmd["ca"].median(), 2)

# std() / mean() : la dispersion rapportee au niveau. Au-dessus de 1,
# l'ecart typique depasse la valeur moyenne elle-meme.
cv = round(cmd["ca"].std() / cmd["ca"].mean(), 2)

print("ecart moyenne-mediane :", ecart_ca)
print("dispersion relative   :", cv)

# Une phrase possible :
# "La commande typique est de 356 EUR (mediane). La moyenne de 590 EUR est
#  tiree par une minorite de tres grosses commandes : 10 % d'entre elles
#  font 41 % du chiffre d'affaires. Un seuil commercial doit se caler sur
#  la mediane, pas sur la moyenne."

In [ ]:
verifier("10a - ecart moyenne-mediane", ecart_ca == 233.84,
         "mean() moins median()")
verifier("10b - dispersion relative", cv == 1.56,
         "std() divise par mean() ; au-dessus de 1, il n'y a pas de valeur typique")